#### Importando librerias y dataset

In [190]:
# Importando librerias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [191]:
# 1) Cargar y limpieza básica
obesidad = pd.read_csv("Obesidad.csv")
obesidad_recortado = obesidad[['YearStart','LocationAbbr','Topic','Question','Data_Value','Stratification1','StratificationCategory1']]

In [192]:
obesidad_recortado['Topic'].unique()

array(['Obesity / Weight Status', 'Fruits and Vegetables - Behavior',
       'Physical Activity - Behavior'], dtype=object)

In [ ]:
# 3. Transformar de formato largo a formato ancho
df_pivot = obesidad_recortado.pivot_table(
    index=['YearStart', 'LocationAbbr', 'StratificationCategory1', 'Stratification1'],
    columns='Question',
    values='Data_Value'
).reset_index() # Reseteamos el índice para que las columnas vuelvan a ser normales

df_pivot.columns.name = None

# Quitando columnas que tienen muchos nulos
df_pivot.drop(columns=["Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)"], inplace=True)
df_pivot.drop(columns=["Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic physical activity and engage in muscle-strengthening activities on 2 or more days a week"], inplace=True)
df_pivot.drop(columns=["Percent of adults who achieve at least 300 minutes a week of moderate-intensity aerobic physical activity or 150 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)"], inplace=True)
df_pivot.drop(columns=["Percent of adults who engage in muscle-strengthening activities on 2 or more days a week"], inplace=True)
df_pivot.drop(columns=["Percent of adults who report consuming fruit less than one time daily"], inplace=True)
df_pivot.drop(columns=["Percent of adults who report consuming vegetables less than one time daily"], inplace=True)

df_pivot.dropna(inplace=True)

In [194]:
df_dummies = pd.get_dummies(df_pivot, dtype=int)
df_dummies

,YearStart,Percent of adults aged 18 years and older who have an overweight classification,Percent of adults aged 18 years and older who have obesity,Percent of adults who engage in no leisure-time physical activity,LocationAbbr_AK,LocationAbbr_AL,LocationAbbr_AR,LocationAbbr_AZ,LocationAbbr_CA,LocationAbbr_CO,...,Stratification1_High school graduate,Stratification1_Hispanic,"Stratification1_Less than $15,000",Stratification1_Less than high school,Stratification1_Male,Stratification1_Non-Hispanic Black,Stratification1_Non-Hispanic White,Stratification1_Other,Stratification1_Some college or technical school,Stratification1_Total
0,2011,32.0,19.8,16.1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2011,38.7,23.5,18.1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2011,38.9,29.5,21.1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2011,43.3,29.2,24.7,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2011,38.9,33.4,26.0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8147,2016,31.5,25.6,30.8,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8148,2016,31.1,27.7,32.2,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
8150,2016,39.4,29.3,28.3,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
8151,2016,36.8,27.1,22.1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


#### Seleccionando los datos X, Y

In [195]:
# Cogeremos X, Y teniendo en cuenta que queremos predecir el valor de obesidad
X = df_dummies.drop(columns=['Percent of adults aged 18 years and older who have obesity'])
y = df_dummies['Percent of adults aged 18 years and older who have obesity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Entrenamos el modelo de Random Forest Regressor
model = RandomForestRegressor(random_state=42, n_estimators=4000, n_jobs=-1)
model.fit(X_train, y_train)

# Hacemos predicciones
y_pred = model.predict(X_test)

# Evaluamos el modelo
print("Accuracy Random Forest:", model.score(X_test, y_test))

Accuracy: 0.7596491762482045


In [ ]:
# Entrenamos el modelo de regresion lineal
from sklearn.linear_model import LinearRegression

model = LinearRegression(n_jobs=-1)
model.fit(X_train, y_train)

# Hacemos predicciones
y_pred = model.predict(X_test)

# Evaluamos el modelo
print("Accuracy LinearRegression:", model.score(X_test, y_test))

Accuracy: 0.7833228557983495


In [ ]:
# Entrenamos con SVR
from sklearn.svm import SVR

model = SVR(kernel='rbf', C=100, gamma=0.01, epsilon=1.25)
model.fit(X_train, y_train)

# Hacemos predicciones
y_pred = model.predict(X_test)

# Evaluamos el modelo
print("Accuracy SVR:", model.score(X_test, y_test))

Accuracy: 0.7765551947597407
